In [1]:
from pathlib import Path
import os
import random
import json
import pickle
from collections import Counter

import numpy as np
import pandas as pd

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

/home/jamyoung/miniconda3/envs/yolov3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ROOT = Path("../preprocess/preprocessing")
META_PATH = ROOT / "metadata.csv"

classes = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
label2idx = {c: i for i, c in enumerate(classes)}
idx2label = {i: c for c, i in label2idx.items()}

seed = 42
model_name = "tf_efficientnetv2_m"

image_size = 300
batch_size = 8
num_workers = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("device:", device)
print("root exists:", ROOT.exists())
print("metadata exists:", META_PATH.exists())

device: cuda
root exists: True
metadata exists: True


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(seed)

In [4]:
df = pd.read_csv(META_PATH)

print(df.head())
print(df.columns)
print(df.shape)
print(df["dx"].value_counts())

required_cols = [
    "lesion_id",
    "image_id",
    "dx",
    "dx_type",
    "age",
    "sex",
    "localization",
    "dataset"
]

for col in required_cols:
    assert col in df.columns, f"Missing column: {col}"

def get_image_path(row):
    cls = row["dx"]
    image_id = row["image_id"]
    path = ROOT / cls / "enhanced" / f"{image_id}.jpg"
    if path.exists():
        return str(path)
    return None

df["image_path"] = df.apply(get_image_path, axis=1)

missing = df["image_path"].isna().sum()
print("missing images:", missing)

if missing > 0:
    missing_df = df[df["image_path"].isna()][["image_id", "dx", "image_path"]]
    display(missing_df.head())
    raise FileNotFoundError(f"{missing} images missing. Check image paths.")

df["label"] = df["dx"].map(label2idx).astype(int)

print(df[["image_id", "dx", "label", "image_path"]].head())
print(df["label"].value_counts().sort_index())

     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp   
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear   

        dataset  
0  vidir_modern  
1  vidir_modern  
2  vidir_modern  
3  vidir_modern  
4  vidir_modern  
Index(['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization',
       'dataset'],
      dtype='object')
(10015, 8)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
missing images: 0
       image_id   dx  label                                         image_path
0  ISIC_0027419  bkl      2  ../preprocess/preprocessing/bkl/enhanced/ISIC_...
1  ISIC_0025030  bkl      2  ../prepr

In [5]:
train_df, val_df = train_test_split(
    df,
    test_size=0.1,
    stratify=df["dx"],
    random_state=seed
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("train:", len(train_df))
print(train_df["dx"].value_counts())

print("\nval:", len(val_df))
print(val_df["dx"].value_counts())

train: 9013
dx
nv       6034
mel      1002
bkl       989
bcc       463
akiec     294
vasc      128
df        103
Name: count, dtype: int64

val: 1002
dx
nv       671
mel      111
bkl      110
bcc       51
akiec     33
vasc      14
df        12
Name: count, dtype: int64


In [6]:
train_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.03
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [7]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import timm

EFF_META_DIR = Path("outputs_effnetv2_m_metadata_finetune_classifier")
EFF_META_DIR.mkdir(parents=True, exist_ok=True)

num_classes = len(classes)

print("model_name:", model_name)
print("device:", device)
print("train size:", len(train_df))
print("val size:", len(val_df))
print("classes:", classes)

model_name: tf_efficientnetv2_m
device: cuda
train size: 9013
val size: 1002
classes: ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']


In [8]:
train_df_meta = train_df.copy()
val_df_meta = val_df.copy()

# age 缺失值：用训练集 median
age_median = train_df_meta["age"].median()

train_df_meta["age"] = train_df_meta["age"].fillna(age_median)
val_df_meta["age"] = val_df_meta["age"].fillna(age_median)

# sex/localization 缺失值：填 unknown
for col in ["sex", "localization"]:
    train_df_meta[col] = train_df_meta[col].fillna("unknown").astype(str)
    val_df_meta[col] = val_df_meta[col].fillna("unknown").astype(str)

# age 标准化
scaler = StandardScaler()
train_age = scaler.fit_transform(train_df_meta[["age"]])
val_age = scaler.transform(val_df_meta[["age"]])

# sex/localization one-hot
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

train_cat = encoder.fit_transform(train_df_meta[["sex", "localization"]])
val_cat = encoder.transform(val_df_meta[["sex", "localization"]])

# 拼接 metadata
train_meta = np.concatenate([train_age, train_cat], axis=1).astype("float32")
val_meta = np.concatenate([val_age, val_cat], axis=1).astype("float32")

metadata_dim = train_meta.shape[1]

print("metadata_dim:", metadata_dim)
print("train_meta:", train_meta.shape)
print("val_meta:", val_meta.shape)
print("encoder categories:", encoder.categories_)

metadata_dim: 19
train_meta: (9013, 19)
val_meta: (1002, 19)
encoder categories: [array(['female', 'male', 'unknown'], dtype=object), array(['abdomen', 'acral', 'back', 'chest', 'ear', 'face', 'foot',
       'genital', 'hand', 'lower extremity', 'neck', 'scalp', 'trunk',
       'unknown', 'upper extremity'], dtype=object)]


In [9]:
import json

metadata_info = {
    "metadata_dim": int(metadata_dim),
    "age_median": float(age_median),
    "sex_categories": encoder.categories_[0].tolist(),
    "localization_categories": encoder.categories_[1].tolist(),
}

with open(EFF_META_DIR / "metadata_info.json", "w", encoding="utf-8") as f:
    json.dump(metadata_info, f, indent=2, ensure_ascii=False)

metadata_info

{'metadata_dim': 19,
 'age_median': 50.0,
 'sex_categories': ['female', 'male', 'unknown'],
 'localization_categories': ['abdomen',
  'acral',
  'back',
  'chest',
  'ear',
  'face',
  'foot',
  'genital',
  'hand',
  'lower extremity',
  'neck',
  'scalp',
  'trunk',
  'unknown',
  'upper extremity']}

In [10]:
class HAMImageMetadataDataset(Dataset):
    def __init__(self, dataframe, metadata_array, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.metadata_array = metadata_array
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        meta = torch.tensor(self.metadata_array[idx], dtype=torch.float32)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        image_id = row["image_id"]

        return image, meta, label, image_id

In [12]:
train_dataset_meta = HAMImageMetadataDataset(
    train_df_meta,
    train_meta,
    transform=train_tfms
)

val_dataset_meta = HAMImageMetadataDataset(
    val_df_meta,
    val_meta,
    transform=val_tfms
)

train_loader_meta = DataLoader(
    train_dataset_meta,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader_meta = DataLoader(
    val_dataset_meta,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

batch = next(iter(train_loader_meta))
images, metas, labels, image_ids = batch

print("images:", images.shape)
print("metas:", metas.shape)
print("labels:", labels.shape)
print("image_ids:", image_ids[:3])

images: torch.Size([8, 3, 300, 300])
metas: torch.Size([8, 19])
labels: torch.Size([8])
image_ids: ['ISIC_0028345', 'ISIC_0031165', 'ISIC_0028873']


In [11]:
class_counts = train_df_meta["label"].value_counts().sort_index().values
N = class_counts.sum()
C = num_classes

class_weights = N / (C * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32)

for cls, count, weight in zip(classes, class_counts, class_weights):
    print(f"{cls:6s} count={count:5d}, weight={weight.item():.4f}")

pd.DataFrame({
    "class": classes,
    "count": class_counts,
    "weight": class_weights.numpy()
}).to_csv(EFF_META_DIR / "class_weights.csv", index=False)

akiec  count=  294, weight=4.3795
bcc    count=  463, weight=2.7809
bkl    count=  989, weight=1.3019
df     count=  103, weight=12.5007
mel    count= 1002, weight=1.2850
nv     count= 6034, weight=0.2134
vasc   count=  128, weight=10.0592


In [14]:
class EfficientNetV2MetadataClassifier(nn.Module):
    def __init__(
        self,
        metadata_dim,
        model_name="tf_efficientnetv2_m",
        num_classes=7,
        pretrained=True
    ):
        super().__init__()

        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool="avg"
        )

        image_feature_dim = self.backbone.num_features

        self.metadata_mlp = nn.Sequential(
            nn.Linear(metadata_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

        self.classifier = nn.Sequential(
            nn.Linear(image_feature_dim + 64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),

            nn.Linear(512, num_classes)
        )

    def forward(self, image, metadata):
        image_feat = self.backbone(image)
        meta_feat = self.metadata_mlp(metadata)

        feat = torch.cat([image_feat, meta_feat], dim=1)
        logits = self.classifier(feat)

        return logits

In [15]:
eff_meta_model = EfficientNetV2MetadataClassifier(
    metadata_dim=metadata_dim,
    model_name=model_name,
    num_classes=num_classes,
    pretrained=True
).to(device)

criterion_meta = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer_meta = torch.optim.AdamW(
    eff_meta_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler_meta = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_meta,
    mode="max",
    factor=0.5,
    patience=3
)

print("backbone feature dim:", eff_meta_model.backbone.num_features)

backbone feature dim: 1280


In [16]:
eff_meta_model.eval()

with torch.no_grad():
    images = images.to(device)
    metas = metas.to(device)
    labels = labels.to(device)

    logits = eff_meta_model(images, metas)
    loss = criterion_meta(logits, labels)

print("logits:", logits.shape)
print("loss:", loss.item())

logits: torch.Size([8, 7])
loss: 1.9565653800964355


In [17]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

In [18]:
def train_one_epoch_meta(model, loader, optimizer, criterion, device):
    model.train()

    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, metas, labels, _ in tqdm(loader, desc="Training EfficientNetV2 + Metadata"):
        images = images.to(device)
        metas = metas.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images, metas)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = avg_loss

    return metrics


@torch.no_grad()
def evaluate_meta(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    all_image_ids = []

    for images, metas, labels, image_ids in tqdm(loader, desc="Validating EfficientNetV2 + Metadata"):
        images = images.to(device)
        metas = metas.to(device)
        labels = labels.to(device)

        logits = model(images, metas)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        total_loss += loss.item() * images.size(0)

        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        all_probs.extend(probs.detach().cpu().numpy())
        all_image_ids.extend(list(image_ids))

    avg_loss = total_loss / len(loader.dataset)

    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = avg_loss

    return (
        metrics,
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
        all_image_ids
    )

In [18]:
train_metrics = train_one_epoch_meta(
    eff_meta_model,
    train_loader_meta,
    optimizer_meta,
    criterion_meta,
    device
)

val_metrics, y_val_meta, y_pred_meta, y_prob_meta, val_image_ids_meta = evaluate_meta(
    eff_meta_model,
    val_loader_meta,
    criterion_meta,
    device
)

print("train:", train_metrics)
print("val:", val_metrics)
print("pred distribution:", Counter(y_pred_meta))

Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:28<00:00,  4.43it/s]

train: {'accuracy': 0.6233218684122933, 'balanced_accuracy': np.float64(0.5292687864222178), 'precision_macro': 0.37999921062664466, 'recall_macro': 0.5292687864222178, 'f1_macro': 0.4214920811099228, 'loss': 1.2535064863693062}
val: {'accuracy': 0.7405189620758483, 'balanced_accuracy': np.float64(0.7135494806937155), 'precision_macro': 0.522374703218859, 'recall_macro': 0.7135494806937155, 'f1_macro': 0.5637790161598785, 'loss': 0.8878115751667175}
pred distribution: Counter({np.int64(5): 605, np.int64(2): 132, np.int64(4): 70, np.int64(0): 64, np.int64(1): 52, np.int64(3): 44, np.int64(6): 35})


In [19]:
num_epochs = 5
patience = 3
best_f1 = -1.0
no_improve = 0
history = []

for epoch in range(1, num_epochs + 1):
    print("\n" + "=" * 80)
    print(f"Epoch {epoch}/{num_epochs}")
    print("=" * 80)

    train_metrics = train_one_epoch_meta(
        eff_meta_model,
        train_loader_meta,
        optimizer_meta,
        criterion_meta,
        device
    )

    val_metrics, y_val_meta, y_pred_meta, y_prob_meta, val_image_ids_meta = evaluate_meta(
        eff_meta_model,
        val_loader_meta,
        criterion_meta,
        device
    )

    scheduler_meta.step(val_metrics["f1_macro"])

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "lr": optimizer_meta.param_groups[0]["lr"]
    }

    history.append(row)
    print(row)
    print("pred distribution:", Counter(y_pred_meta))

    history_df = pd.DataFrame(history)
    history_df.to_csv(EFF_META_DIR / "training_history.csv", index=False)

    current_f1 = val_metrics["f1_macro"]

    if current_f1 > best_f1:
        best_f1 = current_f1
        no_improve = 0

        torch.save(
            {
                "model_state_dict": eff_meta_model.state_dict(),
                "backbone_state_dict": eff_meta_model.backbone.state_dict(),
                "classes": classes,
                "label2idx": label2idx,
                "idx2label": idx2label,
                "model_name": model_name,
                "image_size": image_size,
                "metadata_dim": metadata_dim,
                "age_median": age_median,
                "class_weights": class_weights,
            },
            EFF_META_DIR / "best_effnetv2_metadata_classifier.pth"
        )

        pd.DataFrame([val_metrics]).to_csv(
            EFF_META_DIR / "metrics.csv",
            index=False
        )

        pred_df = pd.DataFrame({
            "image_id": val_image_ids_meta,
            "true_label": [idx2label[i] for i in y_val_meta],
            "pred_label": [idx2label[i] for i in y_pred_meta],
            "correct": y_val_meta == y_pred_meta
        })

        for i, cls in idx2label.items():
            pred_df[f"prob_{cls}"] = y_prob_meta[:, i]

        pred_df.to_csv(EFF_META_DIR / "predictions.csv", index=False)

        report = classification_report(
            y_val_meta,
            y_pred_meta,
            target_names=classes,
            output_dict=True,
            zero_division=0
        )

        pd.DataFrame(report).transpose().to_csv(
            EFF_META_DIR / "per_class_metrics.csv"
        )

        cm = confusion_matrix(y_val_meta, y_pred_meta, labels=list(range(num_classes)))
        pd.DataFrame(cm, index=classes, columns=classes).to_csv(
            EFF_META_DIR / "confusion_matrix.csv"
        )

        print(f"Saved best EfficientNetV2 + Metadata model. val_f1_macro={best_f1:.4f}")

    else:
        no_improve += 1
        print(f"No improvement: {no_improve}/{patience}")

    if no_improve >= patience:
        print("Early stopping triggered.")
        break


Epoch 1/5


Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:25<00:00,  5.01it/s]


{'epoch': 1, 'train_accuracy': 0.7057583490513702, 'train_balanced_accuracy': np.float64(0.6828485156650548), 'train_precision_macro': 0.49476062348989486, 'train_recall_macro': 0.6828485156650548, 'train_f1_macro': 0.5563677647363898, 'train_loss': 0.9024504510700061, 'val_accuracy': 0.7504990019960079, 'val_balanced_accuracy': np.float64(0.749890914596797), 'val_precision_macro': 0.5833673351138229, 'val_recall_macro': 0.749890914596797, 'val_f1_macro': 0.6187899768308366, 'val_loss': 0.8773177336267532, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 584, np.int64(2): 195, np.int64(0): 80, np.int64(4): 51, np.int64(1): 46, np.int64(6): 26, np.int64(3): 20})
Saved best EfficientNetV2 + Metadata model. val_f1_macro=0.6188

Epoch 2/5


Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:25<00:00,  5.02it/s]


{'epoch': 2, 'train_accuracy': 0.7383778985909242, 'train_balanced_accuracy': np.float64(0.7340834055811112), 'train_precision_macro': 0.5622738483249506, 'train_recall_macro': 0.7340834055811112, 'train_f1_macro': 0.6255513679379929, 'train_loss': 0.8173281420524438, 'val_accuracy': 0.7405189620758483, 'val_balanced_accuracy': np.float64(0.791562321146286), 'val_precision_macro': 0.5509540206555811, 'val_recall_macro': 0.791562321146286, 'val_f1_macro': 0.6182153981816133, 'val_loss': 0.7699470397958734, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 518, np.int64(4): 161, np.int64(2): 140, np.int64(1): 63, np.int64(3): 42, np.int64(0): 41, np.int64(6): 37})
No improvement: 1/3

Epoch 3/5


...

{'epoch': 3, 'train_accuracy': 0.7627870853212028, 'train_balanced_accuracy': np.float64(0.7586448950078923), 'train_precision_macro': 0.6042530841245192, 'train_recall_macro': 0.7586448950078923, 'train_f1_macro': 0.6642961829118923, 'train_loss': 0.7280585263699415, 'val_accuracy': 0.8153692614770459, 'val_balanced_accuracy': np.float64(0.7696304952882994), 'val_precision_macro': 0.6813737983263758, 'val_recall_macro': 0.7696304952882994, 'val_f1_macro': 0.7112458933114066, 'val_loss': 0.7544864272753831, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 624, np.int64(2): 142, np.int64(4): 87, np.int64(0): 65, np.int64(1): 55, np.int64(6): 15, np.int64(3): 14})
Saved best EfficientNetV2 + Metadata model. val_f1_macro=0.7112

Epoch 4/5


Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:25<00:00,  4.94it/s]


{'epoch': 4, 'train_accuracy': 0.7807611228225896, 'train_balanced_accuracy': np.float64(0.7866306623765912), 'train_precision_macro': 0.6309295529485065, 'train_recall_macro': 0.7866306623765912, 'train_f1_macro': 0.6927178800847068, 'train_loss': 0.6413885311249503, 'val_accuracy': 0.7365269461077845, 'val_balanced_accuracy': np.float64(0.7927846298627398), 'val_precision_macro': 0.5432110881252904, 'val_recall_macro': 0.7927846298627398, 'val_f1_macro': 0.6065165333718213, 'val_loss': 0.8277098008388739, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 531, np.int64(4): 144, np.int64(0): 106, np.int64(2): 95, np.int64(1): 63, np.int64(6): 33, np.int64(3): 30})
No improvement: 1/3

Epoch 5/5


Validating EfficientNetV2 + Metadata: 100%|██████████| 126/126 [00:25<00:00,  4.89it/s]


{'epoch': 5, 'train_accuracy': 0.788749583934317, 'train_balanced_accuracy': np.float64(0.8043914886888627), 'train_precision_macro': 0.6558491035298868, 'train_recall_macro': 0.8043914886888627, 'train_f1_macro': 0.7165412405189443, 'train_loss': 0.6027027334752152, 'val_accuracy': 0.8423153692614771, 'val_balanced_accuracy': np.float64(0.8449492319584619), 'val_precision_macro': 0.708643356448882, 'val_recall_macro': 0.8449492319584619, 'val_f1_macro': 0.7573813350903633, 'val_loss': 0.5490942888273574, 'lr': 0.0001}
pred distribution: Counter({np.int64(5): 633, np.int64(4): 119, np.int64(2): 91, np.int64(1): 63, np.int64(0): 59, np.int64(3): 21, np.int64(6): 16})
Saved best EfficientNetV2 + Metadata model. val_f1_macro=0.7574


In [12]:
print("model_name:", model_name)
print("device:", device)
print("image_size:", image_size)
print("batch_size:", batch_size)
print("metadata_dim:", metadata_dim)

print("train_df_meta:", train_df_meta.shape)
print("val_df_meta:", val_df_meta.shape)

print("train_meta:", train_meta.shape)
print("val_meta:", val_meta.shape)

print("checkpoint dir:", EFF_META_DIR)
print("checkpoint exists:", (EFF_META_DIR / "best_effnetv2_metadata_classifier.pth").exists())

model_name: tf_efficientnetv2_m
device: cuda
image_size: 300
batch_size: 8
metadata_dim: 19
train_df_meta: (9013, 10)
val_df_meta: (1002, 10)
train_meta: (9013, 19)
val_meta: (1002, 19)
checkpoint dir: outputs_effnetv2_m_metadata_finetune_classifier
checkpoint exists: True


In [13]:
EFF_HBP_RF_DIR = Path("outputs_effnetv2_m_finetuned_hbp_metadata_rf")
EFF_HBP_RF_DIR.mkdir(parents=True, exist_ok=True)

print(EFF_HBP_RF_DIR)

outputs_effnetv2_m_finetuned_hbp_metadata_rf


In [14]:
extract_tfms = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class HAMImageMetadataExtractDataset(Dataset):
    def __init__(self, dataframe, metadata_array, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.metadata_array = metadata_array
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        meta = torch.tensor(self.metadata_array[idx], dtype=torch.float32)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        image_id = row["image_id"]

        return image, meta, label, image_id

In [15]:
extract_batch_size = batch_size

train_extract_dataset = HAMImageMetadataExtractDataset(
    train_df_meta,
    train_meta,
    transform=extract_tfms
)

val_extract_dataset = HAMImageMetadataExtractDataset(
    val_df_meta,
    val_meta,
    transform=extract_tfms
)

train_extract_loader = DataLoader(
    train_extract_dataset,
    batch_size=extract_batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

val_extract_loader = DataLoader(
    val_extract_dataset,
    batch_size=extract_batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

batch = next(iter(train_extract_loader))
images, metas, labels, image_ids = batch

print("images:", images.shape)
print("metas:", metas.shape)
print("labels:", labels.shape)
print("image_ids:", image_ids[:3])

images: torch.Size([8, 3, 300, 300])
metas: torch.Size([8, 19])
labels: torch.Size([8])
image_ids: ['ISIC_0026337', 'ISIC_0028005', 'ISIC_0029667']


In [16]:
def bilinear_pooling(f1, f2):
    """
    f1, f2: [B, D, H, W]
    return: [B, D*D]
    """
    B, D, H, W = f1.shape

    x = f1.view(B, D, H * W)
    y = f2.view(B, D, H * W)

    bilinear = torch.bmm(x, y.transpose(1, 2)) / (H * W)
    bilinear = bilinear.view(B, D * D)

    return bilinear


def normalize_bilinear_features(x, eps=1e-8):
    """
    signed sqrt + L2 normalization
    """
    x = torch.sign(x) * torch.sqrt(torch.abs(x) + eps)
    x = F.normalize(x, p=2, dim=1)
    return x

In [17]:
class FineTunedHBPFeatureExtractor(nn.Module):
    def __init__(
        self,
        model_name="tf_efficientnetv2_m",
        out_indices=(2, 3, 4),
        proj_dim=64,
        pretrained=False
    ):
        super().__init__()

        self.feature_extractor = timm.create_model(
            model_name,
            pretrained=pretrained,
            features_only=True,
            out_indices=out_indices
        )

        channels = self.feature_extractor.feature_info.channels()

        self.projections = nn.ModuleList([
            nn.Conv2d(c, proj_dim, kernel_size=1)
            for c in channels
        ])

        self.proj_dim = proj_dim
        self.out_indices = out_indices

    def forward(self, x):
        feats = self.feature_extractor(x)

        target_size = feats[-1].shape[-2:]

        projected = []

        for feat, proj in zip(feats, self.projections):
            z = proj(feat)
            z = F.interpolate(
                z,
                size=target_size,
                mode="bilinear",
                align_corners=False
            )
            projected.append(z)

        f1, f2, f3 = projected

        b12 = bilinear_pooling(f1, f2)
        b23 = bilinear_pooling(f2, f3)
        b13 = bilinear_pooling(f1, f3)

        hbp = torch.cat([b12, b23, b13], dim=1)
        hbp = normalize_bilinear_features(hbp)

        return hbp

In [18]:
checkpoint_path = EFF_META_DIR / "best_effnetv2_metadata_classifier.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

print(checkpoint.keys())
print("checkpoint model_name:", checkpoint.get("model_name", None))
print("checkpoint image_size:", checkpoint.get("image_size", None))

/tmp/ipykernel_121057/2136635720.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


dict_keys(['model_state_dict', 'backbone_state_dict', 'classes', 'label2idx', 'idx2label', 'model_name', 'image_size', 'metadata_dim', 'age_median', 'metadata_info', 'class_weights'])
checkpoint model_name: tf_efficientnetv2_m
checkpoint image_size: 300


In [19]:
out_indices = (2, 3, 4)
proj_dim = 64

hbp_model = FineTunedHBPFeatureExtractor(
    model_name=model_name,
    out_indices=out_indices,
    proj_dim=proj_dim,
    pretrained=False
).to(device)

missing, unexpected = hbp_model.feature_extractor.load_state_dict(
    checkpoint["backbone_state_dict"],
    strict=False
)

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

print("First few missing:", missing[:5])
print("First few unexpected:", unexpected[:5])

Missing keys: 0
Unexpected keys: 6
First few missing: []
First few unexpected: ['conv_head.weight', 'bn2.weight', 'bn2.bias', 'bn2.running_mean', 'bn2.running_var']


In [20]:
import torch.nn.functional as F
hbp_model.eval()

with torch.no_grad():
    images = images.to(device)
    feats = hbp_model(images)

print("HBP features:", feats.shape)
print("Expected HBP dim:", 3 * proj_dim * proj_dim)
print("NaN:", torch.isnan(feats).any().item())
print("Inf:", torch.isinf(feats).any().item())

HBP features: torch.Size([8, 12288])
Expected HBP dim: 12288
NaN: False
Inf: False


In [21]:
@torch.no_grad()
def extract_hbp_features(model, loader, device):
    model.eval()

    all_features = []
    all_meta = []
    all_labels = []
    all_image_ids = []

    for images, metas, labels, image_ids in tqdm(loader, desc="Extracting fine-tuned HBP features"):
        images = images.to(device)

        feats = model(images)
        feats = feats.detach().cpu().numpy()

        all_features.append(feats)
        all_meta.append(metas.numpy())
        all_labels.extend(labels.numpy())
        all_image_ids.extend(list(image_ids))

    X = np.concatenate(all_features, axis=0)
    M = np.concatenate(all_meta, axis=0)
    y = np.array(all_labels)

    return X, M, y, all_image_ids

In [22]:
X_train_hbp, M_train, y_train_hbp, train_ids_hbp = extract_hbp_features(
    hbp_model,
    train_extract_loader,
    device
)

X_val_hbp, M_val, y_val_hbp, val_ids_hbp = extract_hbp_features(
    hbp_model,
    val_extract_loader,
    device
)

print("X_train_hbp:", X_train_hbp.shape)
print("X_val_hbp:", X_val_hbp.shape)

print("M_train:", M_train.shape)
print("M_val:", M_val.shape)

print("y_train:", y_train_hbp.shape)
print("y_val:", y_val_hbp.shape)

print("NaN:", np.isnan(X_train_hbp).any(), np.isnan(X_val_hbp).any())
print("Inf:", np.isinf(X_train_hbp).any(), np.isinf(X_val_hbp).any())

Extracting fine-tuned HBP features: 100%|██████████| 126/126 [00:21<00:00,  5.91it/s]

X_train_hbp: (9013, 12288)
X_val_hbp: (1002, 12288)
M_train: (9013, 19)
M_val: (1002, 19)
y_train: (9013,)
y_val: (1002,)
NaN: False False
Inf: False False


In [23]:
np.save(EFF_HBP_RF_DIR / "X_train_hbp.npy", X_train_hbp)
np.save(EFF_HBP_RF_DIR / "X_val_hbp.npy", X_val_hbp)
np.save(EFF_HBP_RF_DIR / "M_train.npy", M_train)
np.save(EFF_HBP_RF_DIR / "M_val.npy", M_val)
np.save(EFF_HBP_RF_DIR / "y_train.npy", y_train_hbp)
np.save(EFF_HBP_RF_DIR / "y_val.npy", y_val_hbp)

pd.DataFrame({"image_id": train_ids_hbp, "label": y_train_hbp}).to_csv(
    EFF_HBP_RF_DIR / "train_feature_ids.csv",
    index=False
)

pd.DataFrame({"image_id": val_ids_hbp, "label": y_val_hbp}).to_csv(
    EFF_HBP_RF_DIR / "val_feature_ids.csv",
    index=False
)

In [24]:
print("HBP mean:", X_train_hbp.mean())
print("HBP std:", X_train_hbp.std())
print("HBP min:", X_train_hbp.min())
print("HBP max:", X_train_hbp.max())

print("Metadata mean:", M_train.mean())
print("Metadata std:", M_train.std())

HBP mean: 1.8096858e-05
HBP std: 0.009021077
HBP min: -0.046211887
HBP max: 0.04535265
Metadata mean: 0.10526314
Metadata std: 0.3831637


- 先不拼 metadata，看看 fine-tuned HBP image feature 本身是否有效。
    - 有效，相关代码已删 

- fine-tuned HBP features 已经有效。
    - balanced_accuracy > 0.6
    - f1_macro > 0.6

- 拼接 metadata，做 HBP + metadata + RF

In [27]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from collections import Counter
import pickle

In [25]:
X_train_fused = np.concatenate([X_train_hbp, M_train], axis=1)
X_val_fused = np.concatenate([X_val_hbp, M_val], axis=1)

print("X_train_fused:", X_train_fused.shape)
print("X_val_fused:", X_val_fused.shape)

X_train_fused: (9013, 12307)
X_val_fused: (1002, 12307)


- Scaler + PCA：

In [28]:
pca_dim_fused = 512

scaler_fused = StandardScaler()
X_train_fused_scaled = scaler_fused.fit_transform(X_train_fused)
X_val_fused_scaled = scaler_fused.transform(X_val_fused)

pca_fused = PCA(
    n_components=pca_dim_fused,
    random_state=seed
)

X_train_fused_pca = pca_fused.fit_transform(X_train_fused_scaled)
X_val_fused_pca = pca_fused.transform(X_val_fused_scaled)

print("X_train_fused_pca:", X_train_fused_pca.shape)
print("explained variance:", pca_fused.explained_variance_ratio_.sum())

: 